# NB9 — Learning-to-Rank

## Objective

Implement a true Learning-to-Rank model for the same problem evaluated in NB8.

The model learns from the LLM-as-a-Judge v4 qrels and combines lexical, semantic, and metadata features:

- `bm25_score`
- `semantic_score`
- `norm_rating`
- `norm_popularity`

The model is a pointwise Ridge ranker. The train/test split is performed **by query**, not by individual row, to prevent leakage between training and test.

### Inputs

NB9 only consumes artifacts produced upstream:

- `data/tokens.pkl` → BM25 tokens produced by NB6
- `data/embeddings_nat.npy` → `Description_nat` embeddings produced by NB7
- `data/evaluation/expanded_qrels_v4.json` → judgments produced by the v4 qrels generator

NB9 **does not generate** tokens, embeddings, or a FAISS index.

### Outputs

The notebook:

1. builds the `(query, game)` dataset using only the BGGIds present in the qrels;
2. builds the features;
3. trains Ridge;
4. evaluates on completely held-out queries;
5. runs 5-fold GroupKFold to obtain a more robust estimate;
6. compares BM25 and LTR;
7. saves the final model to `data/ltr_ridge_model.pkl`.


In [1]:
from pathlib import Path
import ast
import html
import json
import pickle
import string

import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DETAILS_PATH = DATA_DIR / "details.csv"
TOKENS_PATH = DATA_DIR / "tokens.pkl"
EMB_NAT_PATH = DATA_DIR / "embeddings_nat.npy"
QRELS_PATH = DATA_DIR / "evaluation" / "expanded_qrels_v4.json"
MODEL_OUT = DATA_DIR / "ltr_ridge_model.pkl"

assert TOKENS_PATH.exists(), f"Missing {TOKENS_PATH}. Run NB6 first."
assert EMB_NAT_PATH.exists(), f"Missing {EMB_NAT_PATH}. Run NB7 first."
assert QRELS_PATH.exists(), f"Missing {QRELS_PATH}. Generate the v4 qrels first."

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)


/home/tommaso/projects/Board-Game-Data-Analytics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /home/tommaso/projects/Board-Game-Data-Analytics
Data dir: /home/tommaso/projects/Board-Game-Data-Analytics/data


In [2]:
# Load the dataset and build Description_nat

df_clean = pd.read_csv(DATA_DIR / "cleaned_dataset.csv")
df_raw = pd.read_csv(DATA_DIR / "DM1_game_dataset.csv")

df = pd.merge(
    df_clean,
    df_raw[["BGGId", "Description"]],
    on="BGGId",
    how="inner",
).dropna(subset=["Description"]).reset_index(drop=True)

orig = pd.read_csv(DETAILS_PATH).drop_duplicates("id")
m = df[["BGGId"]].merge(
    orig,
    left_on="BGGId",
    right_on="id",
    how="left",
)

assert len(m) == len(df), "Il merge con details.csv ha cambiato il numero di righe."

def clean_html(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(html.unescape(text))
    return " ".join(text.split())

nat = m["description"].map(clean_html)
has_orig = nat.str.len() > 0

df["Description_nat"] = nat.where(has_orig, df["Description"])

print(f"Dataset rows: {len(df)}")
print(f"Original natural descriptions: {int(has_orig.sum())}")
print(f"Fallback to Description: {int((~has_orig).sum())}")

# BGGId must be unique in the dataframe.
if df["BGGId"].duplicated().any():
    raise ValueError("BGGId duplicati nel dataset: impossibile costruire un mapping univoco.")


Dataset rows: 19728
Original natural descriptions: 19256
Fallback to Description: 472


In [3]:
# Load NB6 / NB7 artifacts.
# No artifact is regenerated in NB9.

with open(TOKENS_PATH, "rb") as f:
    tokens = pickle.load(f)

if len(tokens) != len(df):
    raise ValueError(
        f"tokens.pkl contiene {len(tokens)} righe, ma df contiene {len(df)} righe. "
        "Regenerate tokens.pkl by running NB6."
    )

embeddings = np.load(EMB_NAT_PATH).astype("float32")

if embeddings.ndim != 2 or embeddings.shape[0] != len(df):
    raise ValueError(
        f"embeddings_nat.npy ha shape {embeddings.shape}; "
        f"expected ({len(df)}, dim). Regenerate the artifact with NB7."
    )

print("tokens.pkl:", len(tokens), "documents")
print("embeddings_nat.npy:", embeddings.shape)

# The model is loaded only to encode queries.
model = SentenceTransformer("all-MiniLM-L6-v2")

stop_words = set()
try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize
    for pkg in ("punkt", "punkt_tab", "stopwords"):
        nltk.download(pkg, quiet=True)
    stop_words = set(stopwords.words("english"))
except Exception:
    pass

def preprocess_text(text):
    if not isinstance(text, str):
        return []
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    try:
        return [w for w in word_tokenize(text) if w not in stop_words]
    except Exception:
        return [w for w in text.split() if w not in stop_words]

bm25 = BM25Okapi(tokens)


tokens.pkl: 19728 documents
embeddings_nat.npy: (19728, 384)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14367.40it/s]


In [4]:
# Load v4 qrels.
# Qrels BGGIds are the authoritative source for the judged candidate pool.

with open(QRELS_PATH, "r", encoding="utf-8") as f:
    qrels_payload = json.load(f)

full_qrels = qrels_payload["qrels"]
qrels_names = qrels_payload.get("names", {})

print("Queries:", len(full_qrels))
print("Judgments:", sum(len(v) for v in full_qrels.values()))

# Check that every judged BGGId exists in the dataset.
dataset_ids = set(df["BGGId"].astype(int))
qrel_ids = {int(gid) for q in full_qrels.values() for gid in q.keys()}

missing_ids = qrel_ids - dataset_ids
if missing_ids:
    raise ValueError(
        f"{len(missing_ids)} BGGId presenti nei qrels non esistono nel dataset. "
        f"Esempio: {sorted(missing_ids)[:10]}"
    )

# Check v4 buckets: 2 judgments at 3, 6 at 2, 8 at 1, the rest at 0
bucket_errors = []
for q, judgments in full_qrels.items():
    counts = {s: list(judgments.values()).count(s) for s in [0, 1, 2, 3]}
    if counts.get(3, 0) != 2 or counts.get(2, 0) != 6 or counts.get(1, 0) != 8:
        bucket_errors.append((q, counts))

print("Bucket checks:", "OK" if not bucket_errors else f"{len(bucket_errors)} queries with non-standard buckets")


Queries: 100
Judgments: 3547
Bucket checks: OK


In [5]:
# Metric utilities

def calculate_mrr(ranked_ids, qrels):
    for rank, gid in enumerate(ranked_ids, start=1):
        rel = qrels.get(str(gid), qrels.get(gid, 0))
        if rel > 0:
            return 1.0 / rank
    return 0.0

def calculate_ndcg(ranked_ids, qrels, k=5):
    """NDCG@k for graded relevance judgments using exponential gain."""
    ranked = ranked_ids[:k]
    gains = [
        qrels.get(str(gid), qrels.get(gid, 0))
        for gid in ranked
    ]

    dcg = sum(
        (2 ** rel - 1) / np.log2(rank + 1)
        for rank, rel in enumerate(gains, start=1)
    )

    ideal = sorted(
        [int(v) for v in qrels.values()],
        reverse=True
    )[:k]

    idcg = sum(
        (2 ** rel - 1) / np.log2(rank + 1)
        for rank, rel in enumerate(ideal, start=1)
    )

    return dcg / idcg if idcg > 0 else 0.0


In [6]:
# Build features for each query.
#
# Important:
# - the candidate pool is exactly the one present in qrels v4;
# - no new retrieval is performed;
# - semantic_score uses the same L2 geometry as FAISS;
# - BGGId is the canonical document identifier, not Name.

id_to_idx = pd.Series(df.index.values, index=df["BGGId"].astype(int))

def get_numeric_column(candidates, default=0.0):
    for col in candidates:
        if col in df.columns:
            return pd.to_numeric(df[col], errors="coerce").fillna(default)
    return pd.Series(default, index=df.index, dtype=float)

rating_series = get_numeric_column(["Rating", "rating"])
popularity_series = get_numeric_column(["Avg_popularity_log", "avg_popularity_log", "Popularity"])

def extract_query_features(query, query_qrels):
    judged_ids = [int(gid) for gid in query_qrels.keys()]
    indices = id_to_idx.reindex(judged_ids)

    if indices.isna().any():
        missing = indices[indices.isna()].index.tolist()
        raise ValueError(f"Missing BGGIds for query '{query}': {missing[:10]}")

    indices = indices.astype(int).to_numpy()

    pool = df.iloc[indices].copy()
    pool["BGGId"] = pool["BGGId"].astype(int)

    # Lexical feature
    bm25_raw_all = bm25.get_scores(preprocess_text(query))
    pool["bm25_raw"] = bm25_raw_all[indices]

    # Semantic feature: Euclidean distance, consistent with the FAISS L2
    # retrieval used in NB7/NB8 and in qrels v4. Higher similarity means
    # smaller distance, so we negate the distance before MinMax scaling.
    qvec = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=False,
    ).astype("float32")[0]

    docvecs = embeddings[indices]
    l2_distances = np.linalg.norm(docvecs - qvec, axis=1)
    pool["semantic_raw"] = -l2_distances

    # Per-query normalization: features are put on comparable scales.
    def minmax(series):
        lo, hi = series.min(), series.max()
        if hi == lo:
            return np.zeros(len(series), dtype=float)
        return ((series - lo) / (hi - lo)).to_numpy()

    pool["bm25_score"] = minmax(pool["bm25_raw"])
    pool["semantic_score"] = minmax(pool["semantic_raw"])
    pool["norm_rating"] = minmax(rating_series.iloc[indices])
    pool["norm_popularity"] = minmax(popularity_series.iloc[indices])

    # Target: v4 qrels
    pool["relevance_label"] = [
        int(query_qrels[str(gid)])
        for gid in pool["BGGId"]
    ]

    pool["query"] = query

    return pool[
        [
            "query",
            "BGGId",
            "Name",
            "bm25_score",
            "semantic_score",
            "norm_rating",
            "norm_popularity",
            "relevance_label",
        ]
    ]

all_data = [
    extract_query_features(query, qrels)
    for query, qrels in full_qrels.items()
]

ltr_dataset = pd.concat(all_data, ignore_index=True)

expected_rows = sum(len(v) for v in full_qrels.values())
assert len(ltr_dataset) == expected_rows

print(f"LTR dataset: {len(ltr_dataset)} (query, game) pairs")
print(f"Unique queries: {ltr_dataset['query'].nunique()}")
print(f"Unique games: {ltr_dataset['BGGId'].nunique()}")


LTR dataset: 3547 (query, game) pairs
Unique queries: 100
Unique games: 3057


In [7]:
# Preliminary feature checks

features = [
    "bm25_score",
    "semantic_score",
    "norm_rating",
    "norm_popularity",
]

print(ltr_dataset[features + ["relevance_label"]].describe())

corr = {}
for feature in features:
    values = []
    for _, g in ltr_dataset.groupby("query"):
        if g[feature].nunique() > 1 and g["relevance_label"].nunique() > 1:
            values.append(
                spearmanr(g[feature], g["relevance_label"]).statistic
            )
    corr[feature] = float(np.nanmean(values))

print("\nMean within-query Spearman correlation:")
for feature, value in corr.items():
    print(f"  {feature:18s}: {value:.3f}")


        bm25_score  semantic_score  norm_rating  norm_popularity  \
count  3547.000000     3547.000000  3547.000000      3547.000000   
mean      0.532239        0.576353     0.469270         0.385451   
std       0.284684        0.263169     0.374433         0.252370   
min       0.000000        0.000000     0.000000         0.000000   
25%       0.324453        0.389975     0.000000         0.193473   
50%       0.584253        0.628979     0.500000         0.339241   
75%       0.747008        0.782220     1.000000         0.547344   
max       1.000000        1.000000     1.000000         1.000000   

       relevance_label  
count      3547.000000  
mean          0.733014  
std           0.934119  
min           0.000000  
25%           0.000000  
50%           0.000000  
75%           1.000000  
max           3.000000  

Mean within-query Spearman correlation:
  bm25_score        : 0.201
  semantic_score    : 0.115
  norm_rating       : 0.172
  norm_popularity   : 0.179


In [8]:
# Train/test split by QUERY.
#
# No test query appears in the training set.

queries = list(full_qrels.keys())

train_queries, test_queries = train_test_split(
    queries,
    test_size=0.15,
    random_state=42,
)

train_data = ltr_dataset[ltr_dataset["query"].isin(train_queries)].copy()
test_data = ltr_dataset[ltr_dataset["query"].isin(test_queries)].copy()

print(f"Train queries: {len(train_queries)}")
print(f"Test queries:  {len(test_queries)}")
print(f"Train pairs:   {len(train_data)}")
print(f"Test pairs:    {len(test_data)}")


Train queries: 85
Test queries:  15
Train pairs:   3008
Test pairs:    539


In [9]:
# Pointwise LTR training.
#
# Ridge learns weights from graded 0/1/2/3 judgments.

ranker = Ridge(alpha=1.0)

X_train = train_data[features]
y_train = train_data["relevance_label"]

ranker.fit(X_train, y_train)

print("LEARNED WEIGHTS")
for feature, weight in zip(features, ranker.coef_):
    print(f"  {feature:18s}: {weight:.4f}")
print(f"  intercept           : {ranker.intercept_:.4f}")


LEARNED WEIGHTS
  bm25_score        : 0.9570
  semantic_score    : 0.8208
  norm_rating       : 0.3352
  norm_popularity   : 0.5074
  intercept           : -0.6081


In [10]:
# Evaluation on completely held-out queries.
#
# Compare BM25 and the actual Ridge LTR model.

bm25_mrr = []
bm25_ndcg = []
ltr_mrr = []
ltr_ndcg = []

for query in test_queries:
    qrels = full_qrels[query]
    qdf = test_data[test_data["query"] == query].copy()

    bm25_ranked = (
        qdf.sort_values("bm25_score", ascending=False)
        .head(5)["BGGId"]
        .astype(int)
        .tolist()
    )

    qdf["pred_score"] = ranker.predict(qdf[features])
    ltr_ranked = (
        qdf.sort_values("pred_score", ascending=False)
        .head(5)["BGGId"]
        .astype(int)
        .tolist()
    )

    bm25_mrr.append(calculate_mrr(bm25_ranked, qrels))
    bm25_ndcg.append(calculate_ndcg(bm25_ranked, qrels, k=5))

    ltr_mrr.append(calculate_mrr(ltr_ranked, qrels))
    ltr_ndcg.append(calculate_ndcg(ltr_ranked, qrels, k=5))

print(f"TEST SET RESULTS ({len(test_queries)} HELD-OUT QUERIES)")
print(f"BM25              MRR {np.mean(bm25_mrr):.3f} | NDCG@5 {np.mean(bm25_ndcg):.3f}")
print(f"Ridge LTR          MRR {np.mean(ltr_mrr):.3f} | NDCG@5 {np.mean(ltr_ndcg):.3f}")


TEST SET RESULTS (15 HELD-OUT QUERIES)
BM25              MRR 0.900 | NDCG@5 0.429
Ridge LTR          MRR 0.889 | NDCG@5 0.469


In [11]:
# 5-fold GroupKFold across all 100 queries.
#
# This is the primary evaluation: each fold holds out
# entire queries, preventing leakage between query groups.

def group_cv_eval(feature_list, n_splits=5, alpha=1.0):
    splitter = GroupKFold(n_splits=n_splits)

    mrr_values = []
    ndcg_values = []

    X = ltr_dataset[feature_list]
    y = ltr_dataset["relevance_label"]
    groups = ltr_dataset["query"]

    for fold, (train_idx, test_idx) in enumerate(
        splitter.split(X, y, groups=groups),
        start=1,
    ):
        train_df = ltr_dataset.iloc[train_idx]
        test_df = ltr_dataset.iloc[test_idx]

        model_cv = Ridge(alpha=alpha)
        model_cv.fit(train_df[feature_list], train_df["relevance_label"])

        for query, qdf in test_df.groupby("query"):
            qrels = full_qrels[query]
            qdf = qdf.copy()

            qdf["pred"] = model_cv.predict(qdf[feature_list])
            ranked = (
                qdf.sort_values("pred", ascending=False)
                .head(5)["BGGId"]
                .astype(int)
                .tolist()
            )

            mrr_values.append(calculate_mrr(ranked, qrels))
            ndcg_values.append(calculate_ndcg(ranked, qrels, k=5))

    return np.mean(mrr_values), np.mean(ndcg_values)

# BM25-only baseline using the same grouped folds.
def grouped_baseline():
    mrr_values, ndcg_values = [], []
    splitter = GroupKFold(n_splits=5)
    groups = ltr_dataset["query"]

    for _, test_idx in splitter.split(ltr_dataset, ltr_dataset["relevance_label"], groups=groups):
        test_df = ltr_dataset.iloc[test_idx]

        for query, qdf in test_df.groupby("query"):
            ranked = (
                qdf.sort_values("bm25_score", ascending=False)
                .head(5)["BGGId"]
                .astype(int)
                .tolist()
            )
            qrels = full_qrels[query]
            mrr_values.append(calculate_mrr(ranked, qrels))
            ndcg_values.append(calculate_ndcg(ranked, qrels, k=5))

    return np.mean(mrr_values), np.mean(ndcg_values)

bm25_cv = grouped_baseline()
ltr_cv = group_cv_eval(features)

print("5-FOLD GROUPED CV")
print(f"BM25 baseline : MRR {bm25_cv[0]:.3f} | NDCG@5 {bm25_cv[1]:.3f}")
print(f"Ridge LTR     : MRR {ltr_cv[0]:.3f} | NDCG@5 {ltr_cv[1]:.3f}")
print(f"Delta MRR     : {ltr_cv[0] - bm25_cv[0]:+.3f}")
print(f"Delta NDCG@5  : {ltr_cv[1] - bm25_cv[1]:+.3f}")


5-FOLD GROUPED CV
BM25 baseline : MRR 0.822 | NDCG@5 0.399
Ridge LTR     : MRR 0.894 | NDCG@5 0.502
Delta MRR     : +0.072
Delta NDCG@5  : +0.103


In [12]:
# Ablation study: identify which features actually help.

ablation_sets = {
    "BM25": ["bm25_score"],
    "BM25 + semantic": ["bm25_score", "semantic_score"],
    "BM25 + semantic + rating": [
        "bm25_score", "semantic_score", "norm_rating"
    ],
    "All features": features,
}

print("5-FOLD ABLATION")
for name, feat_set in ablation_sets.items():
    mrr, ndcg = group_cv_eval(feat_set)
    print(f"{name:28s} MRR {mrr:.3f} | NDCG@5 {ndcg:.3f}")


5-FOLD ABLATION
BM25                         MRR 0.822 | NDCG@5 0.399
BM25 + semantic              MRR 0.864 | NDCG@5 0.435
BM25 + semantic + rating     MRR 0.873 | NDCG@5 0.464
All features                 MRR 0.894 | NDCG@5 0.502


In [13]:
# Train the final model on all queries.
#
# This model is not used for CV estimation; it is the final model to save
# and use after evaluation is complete.

final_ranker = Ridge(alpha=1.0)
final_ranker.fit(
    ltr_dataset[features],
    ltr_dataset["relevance_label"],
)

with open(MODEL_OUT, "wb") as f:
    pickle.dump(
        {
            "model": final_ranker,
            "features": features,
            "alpha": 1.0,
            "training_queries": list(full_qrels.keys()),
            "qrels_version": "expanded_qrels_v4",
        },
        f,
    )

print(f"Saved final LTR model to: {MODEL_OUT}")


Saved final LTR model to: /home/tommaso/projects/Board-Game-Data-Analytics/data/ltr_ridge_model.pkl


## Evaluation scope: BM25 baseline vs. LTR

NB9 evaluates BM25 and Ridge LTR on the **same LLM-judged candidate pool** defined by `expanded_qrels_v4.json`. This is a reranking evaluation, not a full-corpus retrieval evaluation.

For each query, the candidate pool consists exclusively of the BGGIds present in the v4 qrels. BM25 scores are computed for the full corpus, but only the judged candidates are retained and ranked for the NB9 baseline. Ridge LTR ranks exactly the same candidates using the learned feature combination.

This makes the BM25 and LTR results directly comparable within NB9. The BM25 results reported in NB8 are a separate full-corpus retrieval baseline and should not be compared numerically with the NB9 BM25 baseline as if they were the same evaluation.

The primary comparison is therefore **BM25 on the judged candidate pool vs. Ridge LTR on the same pool**, evaluated with the same qrels and query-level GroupKFold splits.


## Interpretation

The primary result is the 5-fold GroupKFold evaluation because entire queries are separated between training and test.

The fixed 15% test split is useful as a reproducible sanity check, but it should not be presented alone as definitive evidence.

The methodologically correct comparison is:

`BM25 baseline` vs `Ridge LTR`

on the same held-out queries and the same v4 qrels.

The ablation study also checks whether the improvement actually comes from the semantic and/or metadata features rather than assuming that all features are useful.

NB9 does not regenerate embeddings: `embeddings_nat.npy` remains an artifact produced exclusively by NB7.
